# 07 · Entire NZ grid setup

Supply a compatible NZ grid GDX input, solve every case and period it contains,
and download **one full results solution file**: `nz-grid-…-solution.zip`.
The input supplies the grid, offers, demand, reserve requirements, and other
model data. PySPD uses native SCIP dispatch and HiGHS pricing.

The solution ZIP contains all twelve PySPD CSV reports and their hash-verified
`manifest.json`, plus the run configuration and input case inventory. It preserves
all report rows, including constraints and audit events. This is PySPD's complete
report solution bundle in ZIP format; it is not a GDX output or a serialized
solver model. The network coverage is the coverage of your supplied input.

This notebook uses the `vspd-v5.0.6` input schema and
`vspd-v5.0.6-reserve` formulation. Use a compatible whole-grid input. Consult
the input-schema guide for v3 or SPD v16 files before changing these settings.

## Install input support

In the notebook environment, install `"pyspd[gdx]==0.1.0"`. The notebook discovers
the bundled GDX reader runtime automatically. If you use a separate GAMS runtime,
set `GDX_RUNTIME_DIRECTORY` below to its system directory. This runtime reads the
input; SCIP and HiGHS perform the solve.

Set `INPUT_GDX`, then choose **Restart Kernel and Run All Cells**. A full daily
input may take substantial time and memory. With no input path, the notebook
prints **NOT RUN** and creates no solution. Saved outputs demonstrate only that
unconfigured path; no whole-grid solve is claimed here.


In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, display

assert sys.version_info[:2] == (3, 13), "Select a Python 3.13 notebook kernel"
assert version("pyspd") == "0.1.0", "This example targets pyspd==0.1.0"
print({name: version(name) for name in ("pyspd", "highspy", "pyscipopt")})

{'pyspd': '0.1.0', 'highspy': '1.15.1', 'pyscipopt': '5.7.1'}


In [2]:
import hashlib
import json
import os
from importlib.util import find_spec
from uuid import uuid4
from zipfile import ZIP_DEFLATED, ZipFile

from pyspd.application import ApplicationConfiguration, PyspdApplication
from pyspd.reporting import ReportBundle

INPUT_GDX = os.environ.get("PYSPD_GDX_INPUT", "")  # e.g. "inputs/nz-grid.gdx"
GDX_RUNTIME_DIRECTORY = os.environ.get("PYSPD_GAMS_SYSTEM_DIRECTORY", "")
OUTPUT_ROOT = Path("pyspd-notebook-output")
configured = bool(INPUT_GDX.strip())
if not configured:
    print(
        "NOT RUN: set INPUT_GDX to your entire NZ grid GDX input, then run all cells."
    )

NOT RUN: set INPUT_GDX to your entire NZ grid GDX input, then run all cells.


## Read the entire NZ grid input

Hash and validate the input, then inventory all case-period records. The preview
below is limited to 30 rows for readability; the solve uses the complete inventory
and network. There is no case filter. All supported cases and periods present in
the file are included.


In [3]:
selected = ()
source_hash = None
if configured:
    from pyspd.data import GdxAdapter, SymbolCatalog
    from pyspd.orchestration import DailyCaseSelector

    input_path = Path(INPUT_GDX).expanduser().resolve()
    assert input_path.is_file(), "INPUT_GDX is not a file"
    if GDX_RUNTIME_DIRECTORY:
        system_path = Path(GDX_RUNTIME_DIRECTORY).expanduser().resolve()
    else:
        runtime = find_spec("gamspy_base")
        assert runtime and runtime.submodule_search_locations, (
            'Install "pyspd[gdx]==0.1.0" or set GDX_RUNTIME_DIRECTORY'
        )
        system_path = Path(next(iter(runtime.submodule_search_locations))).resolve()
    assert system_path.is_dir(), "GDX reader runtime directory does not exist"
    with input_path.open("rb") as stream:
        source_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    symbols = GdxAdapter.read(input_path, system_directory=system_path)
    assert symbols.source_sha256 == source_hash
    SymbolCatalog.vspd_v5().validate(symbols)
    selected = DailyCaseSelector().select(symbols)
    assert selected, "The input contains no supported pricing cases"
    inventory = pd.DataFrame(
        {
            "case_id": item.case_id,
            "date_time": item.date_time,
            "trading_period": item.trading_period,
        }
        for item in selected
    )
    display(inventory.head(30))
    print(f"Solving all {len(selected)} case-period records")
    print("Input SHA-256:", source_hash)
    del symbols  # The application reads the validated source for its solve.

## Solve every case and period

An empty `case_ids` tuple tells the application to include all cases. Each run
gets a new results directory. The application writes all twelve reports and the
manifest; the notebook also records the input inventory and exact configuration.


In [4]:
run = None
if configured:
    output = OUTPUT_ROOT / f"nz-grid-{uuid4().hex}"
    configuration = ApplicationConfiguration(
        formulation_id="vspd-v5.0.6-reserve",
        input_schema="vspd-v5.0.6",
        input_path=input_path,
        gams_system_directory=system_path,
        source_sha256=source_hash,
        output_directory=output,
        case_ids=(),  # All cases and periods in the input.
        worker_count=1,
    )
    run = PyspdApplication().run(configuration)
    assert run.result.state.value == "complete", "The full input did not complete"
    configuration_payload = {
        name: str(value) if isinstance(value, Path) else value
        for name in configuration.__dataclass_fields__
        for value in (getattr(configuration, name),)
    }
    (run.output_directory / "run-configuration.json").write_text(
        json.dumps(configuration_payload, indent=2) + "\n", encoding="utf-8"
    )
    inventory.to_csv(run.output_directory / "input-cases.csv", index=False)
    print("Full results directory:", run.output_directory)
else:
    print("NOT RUN: no GDX input supplied.")

NOT RUN: no GDX input supplied.


## Verify the complete results and create the solution file

Check that every input case-period appears in the summary with a successful
status, and verify every report against its manifest. Inspect violation columns
to understand any scarcity or relaxations. Success does not imply zero violations.

The ZIP keeps every report row. `input-cases.csv` and `run-configuration.json`
record what ran. Extract the ZIP and call `ReportBundle.read(extracted_directory)`
to verify and load the reports later. Reading a full report bundle loads its tables
into memory; large daily studies need enough memory for the constraint report.


In [5]:
def create_solution_file(results_directory):
    """Archive complete reports and study metadata; verify archived report hashes."""
    results_directory = Path(results_directory)
    manifest = json.loads((results_directory / "manifest.json").read_text())
    report_files = sorted(manifest["files"])
    assert len(report_files) == 12, "Expected all twelve PySPD reports"
    filenames = report_files + [
        "manifest.json",
        "run-configuration.json",
        "input-cases.csv",
    ]
    assert all((results_directory / name).is_file() for name in filenames)
    solution_file = results_directory.with_name(
        results_directory.name + "-solution.zip"
    )
    with ZipFile(solution_file, "x", compression=ZIP_DEFLATED) as archive:
        for name in filenames:
            archive.write(results_directory / name, arcname=name)
    with ZipFile(solution_file) as archive:
        assert set(archive.namelist()) == set(filenames)
        assert archive.testzip() is None, "Solution archive failed its CRC check"
        for name, expected in manifest["files"].items():
            with archive.open(name) as stream:
                actual = hashlib.file_digest(stream, "sha256").hexdigest()
            assert actual == expected, f"Archived report hash mismatch: {name}"
    return solution_file


solution_file = None
if run is not None:
    bundle = ReportBundle.read(run.output_directory)
    assert len(bundle.tables) == 12
    assert bundle.provenance.source_sha256 == source_hash
    summary = pd.DataFrame(bundle.tables["summary"].rows)
    expected_periods = {(item.case_id, item.date_time) for item in selected}
    actual_periods = set(zip(summary["case_id"], summary["date_time"], strict=True))
    assert actual_periods == expected_periods, "Results do not cover the full input"
    assert len(summary) == len(selected), "Unexpected duplicate or missing results"
    assert set(summary["status_code"]) == {"1"}, "Inspect non-success solve statuses"
    display(summary.head(30))
    display(
        pd.DataFrame(
            {"table": name, "rows": len(table.rows)}
            for name, table in sorted(bundle.tables.items())
        )
    )
    del bundle  # Release the in-memory report tables before packaging.
    solution_file = create_solution_file(run.output_directory)
    print("Full results solution file:", solution_file)
    print(f"Archive size: {solution_file.stat().st_size / 1024**2:,.1f} MiB")
    display(
        FileLink(
            str(solution_file), result_html_prefix="Download full NZ grid solution: "
        )
    )
else:
    print("NOT RUN: no solution file exists until an input solve completes.")

NOT RUN: no solution file exists until an input solve completes.


## Use the solution

The download above is the complete PySPD results bundle for the supplied grid
input. Start with `summary.csv`, then inspect dispatch, branch flows, reserves,
node prices, published prices, constraints, and audit records as needed. Keep the
source GDX alongside the ZIP to reproduce the study; the input itself is not
copied into the archive. The configuration contains local paths, so review those
and the input's data-sharing terms before sharing your results.
